# Quantra — Train on Colab

One universal **PPO** policy built to **repeatedly pass FTMO-style challenges** (2.5% daily target / 4% trailing wall) — not to maximize PnL.

> ⚠️ **Pick a CPU runtime** (Runtime → Change runtime type → CPU, or High-RAM). The locked net is a tiny **3×256 MLP (~145 inputs)** — the bottleneck is CPU env-stepping, not the GPU. The optimizer below will *prove* whether a GPU is worth it (it usually isn't), so you don't waste paid GPU hours.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/monty313/RL-model-trading-bot-ppo-mlp_Claude-.git quantra_repo
%cd quantra_repo

## 2. Install dependencies
Colab already ships torch, numpy, pandas, scikit-learn, matplotlib. We only add the extras.

In [ ]:
!pip install -q gdown pyarrow psutil optuna seaborn statsmodels gymnasium tqdm
# nvidia-ml-py is only needed if you chose a GPU runtime:
import torch
if torch.cuda.is_available():
    !pip install -q nvidia-ml-py

## 3. Mount Google Drive (price data)
Your 4 symbols (MT5 1m, ~5yr) live in the `rl-trading-data` folder. Mounting lets the data loader read them without re-downloading 500 MB each session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Expected path: /content/drive/MyDrive/rl-trading-data/<SYMBOL>_M1_*.csv
!ls -la "/content/drive/MyDrive/rl-trading-data" 2>/dev/null || echo "Adjust the folder path if your Drive layout differs; gdown-by-ID fallback is built into the loader."

## 4. Race the hardware (CPU vs GPU) and size to ~80%
This is the money-saver: it benchmarks both devices on the real four-head workload, picks the faster one (preferring CPU on a near-tie), scales parallel envs to ~80% utilisation, and reports what it measured.

In [ ]:
from quantra.runtime import RuntimeConfig, plan, print_report, UtilizationMonitor, ensure_dirs

ensure_dirs()
cfg = RuntimeConfig()
with UtilizationMonitor(interval=0.25) as mon:
    hw = plan(state_dim=cfg.nominal_state_dim, hw=cfg.hardware)
util = mon.stop()

print_report(hw)
print('Utilisation during race:', util.render())

## 5. Train
The training entry point lands as milestones **M1–M8** are built (data → features → laws → env → agent → reward → curriculum → trainer). It will be:
```python
python -m quantra.learning_system.trainer  # coming as M8 lands
```
Until then, the cell above already proves the repo runs and the optimizer selects the cheapest fast device for your runtime.